## Introduction : Transformers

### Modeles

Hugging Face : GPT2

### Evaluation

- BLEU/ROUGE Score
- BERT Score

### Bechmark

| Modèle                                | BLEU Score | ROUGE-L Score | BERTScore |
|---------------------------------------|------------|----------------|-----------|
| **GPT-2**          | 0.0115     | 0.0952         | 0.8339    |
| **GPT-2 AventIQ**                     | 0.0047     | 0.1063         | 0.8294    |
| **DistilGPT2 yroshan**                     | 0.0033     | 0.0972         |  0.8278    |


## Common code

In [1]:
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling
import re


/home/rickgao/miniconda3/envs/tf-wsl/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-05-09 17:21:46.377434: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-05-09 17:21:46.377497: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-05-09 17:21:46.380099: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-09 17:21:46.637648: I tensorflow/core/platform/cpu_feature_guard.cc:182] This Tensor

In [2]:
# Load dataset and preprocess
df = pd.read_csv("MovieDataThread.csv")
df = df.dropna(subset=["Script"])
df = df.sample(n=1000, random_state=42)


In [3]:
from sklearn.model_selection import train_test_split

# Split the dataset into training and testing sets
train_texts, test_texts = train_test_split(df["Script"].tolist(), test_size=0.2, random_state=42)
dataset = DatasetDict({
    "train": Dataset.from_dict({"Script": train_texts}),
    "test": Dataset.from_dict({"Script": test_texts})
})


## Hugging face Transformer (GPT2)

https://huggingface.co/openai-community/gpt2

### Model

In [5]:
# Load the tokenizer and model from Hugging Face (GPT2)
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained(model_name)


In [6]:
def preprocess_text(text):
    text = re.sub(r"\n+", "\n", text)
    text = text.replace("\n", " NEWLINE_TOKEN ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def tokenize_function(example):
    processed = [preprocess_text(script) for script in example["Script"]]
    encodings = tokenizer(processed, truncation=True, padding="longest", max_length=256)
    encodings["labels"] = encodings["input_ids"].copy()
    return encodings


tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["Script"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


Parameter 'function'=<function tokenize_function at 0x7f4a02ac5a60> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.
100%|██████████| 1/1 [00:16<00:00, 16.97s/ba]


### Train

In [8]:
# Train the model (Arguments)
training_args = TrainingArguments(
    output_dir="./gpt2-script",
    eval_strategy="epoch",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    num_train_epochs=10,
    logging_dir="./logs",
)

In [9]:
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator
)


/tmp/ipykernel_1635/4126412472.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [10]:
# Train the model
trainer.train()

# Save the model
trainer.save_model("./gpt2-script")

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,2.200200,2.047124
2,1.840600,2.064347
3,1.669900,2.114891
4,1.475800,2.192178
5,1.362900,2.275424
6,1.227200,2.379925
7,1.147400,2.460878
8,1.095500,2.538160
9,1.009900,2.589114
10,0.972200,2.617366


### Load model

In [57]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize
from rouge import Rouge
import re
import string

def calculate_bleu(reference, candidate):
    """
    Calculate BLEU score (it measures the similarity between two sentences)
    :param reference: The reference sentence
    :param candidate: The candidate sentence
    :return: The BLEU score
    """
    smoothing_function = SmoothingFunction().method4
    bleu_score = sentence_bleu([reference], candidate, smoothing_function=smoothing_function)
    return bleu_score

def calculate_rouge(reference, candidate):
    """
    Calculate ROUGE-L score (it calculates the longest common subsequence)
    :param reference: The reference sentence
    :param candidate: The candidate sentence
    :return: The ROUGE score
    """
    rouge = Rouge()
    scores = rouge.get_scores(candidate, reference)
    return scores[0]['rouge-l']['f']

def preprocess_text(text):
    """
    Preprocess the text by removing punctuation and converting to lowercase
    :param text: The input text
    :return: The preprocessed text
    """
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    text = text.lower()
    return text

def calculate_scores(reference, candidate):
    """
    Calculate BLEU and ROUGE scores for the given reference and candidate sentences
    :param reference: The reference sentence
    :param candidate: The candidate sentence
    :return: The BLEU and ROUGE scores
    """
    # Preprocess the text
    reference = preprocess_text(reference)
    candidate = preprocess_text(candidate)

    reference_tokens = word_tokenize(reference)
    candidate_tokens = word_tokenize(candidate)

    bleu_score = calculate_bleu(reference_tokens, candidate_tokens)
    rouge_score = calculate_rouge(reference, candidate)

    return bleu_score, rouge_score

In [58]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling
import evaluate
from bert_score import BERTScorer

# Load the model for evaluation
model = GPT2LMHeadModel.from_pretrained("./gpt2-script")
tokenizer = GPT2Tokenizer.from_pretrained("./gpt2-script")


def generate_text(prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


### Evaluation

In [59]:
import re

def script_tokenize(text):
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"\n", " NEWLINE_TOKEN ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip().split()

def custom_tokenize(text):
    first_tokenize = script_tokenize(text)
    entokenized = " ".join(first_tokenize)

    second_tokenize = word_tokenize(entokenized)
    return second_tokenize

In [ ]:
bleu_scores, rouge_scores = [], []
bert_f1, bert_precision, bert_recall = [], [], []


# Augment the SCRIPT_LIMIT_TOKEN_SIZE for better BERT Scores
SCRIPT_LIMIT_TOKEN_SIZE = 300
GENERATE_LIMIT_TOKEN_SIZE = 50

total = len(test_texts)
i = 1

scorer = BERTScorer(lang="en", model_type="roberta-base", rescale_with_baseline=True)

print("Evaluating the model on the test set...")
for full_text in test_texts:
    print(f"Evaluating {i}/{total}...")

    tokenized = tokenizer(full_text, return_tensors="pt", truncation=True,
                          max_length=SCRIPT_LIMIT_TOKEN_SIZE + GENERATE_LIMIT_TOKEN_SIZE)
    input_ids = tokenized.input_ids[0]

    if len(input_ids) <= SCRIPT_LIMIT_TOKEN_SIZE:
        i += 1
        continue

    seed_input_ids = input_ids[:SCRIPT_LIMIT_TOKEN_SIZE]
    expected_output_ids = input_ids[SCRIPT_LIMIT_TOKEN_SIZE:SCRIPT_LIMIT_TOKEN_SIZE + GENERATE_LIMIT_TOKEN_SIZE]

    seed_text = tokenizer.decode(seed_input_ids, skip_special_tokens=True)
    seed_text = custom_tokenize(seed_text)
    seed_text = " ".join(seed_text)
    
    reference_text = tokenizer.decode(expected_output_ids, skip_special_tokens=True).strip()
    reference_text = custom_tokenize(reference_text)
    reference_text = " ".join(reference_text)

    generated_text = generate_text(seed_text, max_new_tokens=GENERATE_LIMIT_TOKEN_SIZE)
    generated_text = re.sub(r"\s+", " ", generated_text).strip()
    generated_text = custom_tokenize(generated_text)
    generated_text = " ".join(generated_text)[len(seed_text) + 1:]

    candidate = generated_text
    reference = reference_text

    print(f"Seed: {seed_text}")
    print(f"Candidate: {candidate}")
    print(f"Reference: {reference}")
    
    bleu_score, rouge_score = calculate_scores(reference, candidate)

    P, R, F1 = scorer.score([candidate], [reference])
    f1_score = F1.mean().item()
    precision_score = P.mean().item()
    recall_score = R.mean().item()

    bleu_scores.append(bleu_score)
    rouge_scores.append(rouge_score)
    bert_f1.append(f1_score)
    bert_precision.append(precision_score)
    bert_recall.append(recall_score)

    print(f"BLEU score: {bleu_score}")
    print(f"ROUGE score: {rouge_score}")
    print(f"BERT F1 score: {f1_score}")
    print(f"BERT Precision score: {precision_score}")
    print(f"BERT Recall score: {recall_score}")

    with open(f"transformers-GPT2-{SCRIPT_LIMIT_TOKEN_SIZE}-{GENERATE_LIMIT_TOKEN_SIZE}.csv", "a") as f:
        f.write(f"{seed_text};{candidate};{reference};{bleu_score};{rouge_score};{f1_score};{precision_score};{recall_score}\n")

    i += 1

# Final scores
print(f"Average BLEU score : {sum(bleu_scores) / len(bleu_scores):.6f}")
print(f"Average ROUGE score : {sum(rouge_scores) / len(rouge_scores):.6f}")
print(f"Average BERT F1 score : {sum(bert_f1) / len(bert_f1):.6f}")
print(f"Average BERT Precision score : {sum(bert_precision) / len(bert_precision):.6f}")
print(f"Average BERT Recall score : {sum(bert_recall) / len(bert_recall):.6f}")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Evaluating the model on the test set...
Evaluating 1/200...
Seed: 1 NEWLINE_TOKEN Delhi flight ! NEWLINE_TOKEN Sorry , Ma'am . NEWLINE_TOKEN Boarding is closed . NEWLINE_TOKEN Listen , I know ! NEWLINE_TOKEN I am so sorry , I got delayed . NEWLINE_TOKEN Just do something . NEWLINE_TOKEN Please help me . NEWLINE_TOKEN Sorry , Ma'am . NEWLINE_TOKEN You should have been on time . NEWLINE_TOKEN I know , I am so sorry ! NEWLINE_TOKEN I got stuck in traffic . NEWLINE_TOKEN You know how it is . NEWLINE_TOKEN Just , please ! Can you do this ? NEWLINE_TOKEN Okay , hold on . NEWLINE_TOKEN Thank you . NEWLINE_TOKEN Window seat , please . NEWLINE_TOKEN Excuse me . NEWLINE_TOKEN `` Oh you ! '' NEWLINE_TOKEN `` Oh you ! '' NEWLINE_TOKEN `` Entry to Delhi . '' NEWLINE_TOKEN `` Oh you ! Entry to Delhi . '' NEWLINE_TOKEN `` Oh you ! Entry to Delhi . Oh welcome . '' NEWLINE_TOKEN `` Delhi is fantastic . NEWLINE_TOKEN Delhi is wonderful . '' NEWLINE_TOKEN `` Delhi is jovial . NEWLINE_TOKEN Delhi is fun l

## TODO

- Augmenter l'epoch
- max_length=128 ou 256

## Hugging face Transformer (AventIQ)

https://huggingface.co/AventIQ-AI/gpt-2-movie-script-writter

In [37]:
from transformers import AutoTokenizer, AutoModelForCausalLM


# Load the tokenizer and model from Hugging Face (GPT2)
model_name = "AventIQ-AI/gpt-2-movie-script-writter"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained(model_name)


In [38]:
def preprocess_text(text):
    text = re.sub(r"\n+", "\n", text)
    text = text.replace("\n", " NEWLINE_TOKEN ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def tokenize_function(example):
    processed = [preprocess_text(script) for script in example["Script"]]
    encodings = tokenizer(processed, truncation=True, padding="longest", max_length=256)
    encodings["labels"] = encodings["input_ids"].copy()
    return encodings

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["Script"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


100%|██████████| 1/1 [00:16<00:00, 16.13s/ba]


### Train

In [42]:
# Train the model (Arguments)
training_args = TrainingArguments(
    output_dir="./aventiq-script",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    num_train_epochs=10,
    logging_dir="./logs",
)

In [43]:
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator
)


/tmp/ipykernel_1635/4126412472.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [44]:
# Train the model
trainer.train()

# Save the model
trainer.save_model("./aventiq-script")

Epoch,Training Loss,Validation Loss
1,1.707500,2.107295
2,1.683300,2.130653
3,1.508700,2.196337
4,1.322000,2.309672
5,1.215900,2.432051
6,1.089500,2.538767
7,1.019000,2.622548
8,0.971700,2.707131
9,0.888100,2.755775
10,0.854300,2.788780


In [45]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling
import evaluate

# Load the model for evaluation
model = GPT2LMHeadModel.from_pretrained("./aventiq-script")
tokenizer = GPT2Tokenizer.from_pretrained("./aventiq-script")

# Generate text

def generate_text(prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


### Evaluation

In [46]:
import re

def script_tokenize(text):
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"\n", " NEWLINE_TOKEN ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip().split()

def custom_tokenize(text):
    first_tokenize = script_tokenize(text)
    entokenized = " ".join(first_tokenize)

    second_tokenize = word_tokenize(entokenized)
    return second_tokenize

In [47]:
import re
from bert_score import BERTScorer

bleu_scores, rouge_scores = [], []
bert_f1, bert_precision, bert_recall = [], [], []

SCRIPT_LIMIT_TOKEN_SIZE = 50
GENERATE_LIMIT_TOKEN_SIZE = 50

total = len(test_texts)
i = 1

scorer = BERTScorer(lang="en", model_type="roberta-base", rescale_with_baseline=True)

print("Evaluating the model on the test set...")
for full_text in test_texts:
    print(f"Evaluating {i}/{total}...")

    tokenized = tokenizer(full_text, return_tensors="pt", truncation=True,
                          max_length=SCRIPT_LIMIT_TOKEN_SIZE + GENERATE_LIMIT_TOKEN_SIZE)
    input_ids = tokenized.input_ids[0]

    if len(input_ids) <= SCRIPT_LIMIT_TOKEN_SIZE:
        i += 1
        continue

    seed_input_ids = input_ids[:SCRIPT_LIMIT_TOKEN_SIZE]
    expected_output_ids = input_ids[SCRIPT_LIMIT_TOKEN_SIZE:SCRIPT_LIMIT_TOKEN_SIZE + GENERATE_LIMIT_TOKEN_SIZE]

    seed_text = tokenizer.decode(seed_input_ids, skip_special_tokens=True)
    seed_text = custom_tokenize(seed_text)
    seed_text = " ".join(seed_text)
    
    reference_text = tokenizer.decode(expected_output_ids, skip_special_tokens=True).strip()
    reference_text = custom_tokenize(reference_text)
    reference_text = " ".join(reference_text)

    generated_text = generate_text(seed_text, max_new_tokens=GENERATE_LIMIT_TOKEN_SIZE)
    generated_text = re.sub(r"\s+", " ", generated_text).strip()
    generated_text = custom_tokenize(generated_text)
    generated_text = " ".join(generated_text)[len(seed_text) + 1:]

    candidate = generated_text
    reference = reference_text

    print(f"Seed: {seed_text}")
    print(f"Candidate: {candidate}")
    print(f"Reference: {reference}")
    
    bleu_score, rouge_score = calculate_scores(reference, candidate)

    P, R, F1 = scorer.score([candidate], [reference])
    f1_score = F1.mean().item()
    precision_score = P.mean().item()
    recall_score = R.mean().item()

    bleu_scores.append(bleu_score)
    rouge_scores.append(rouge_score)
    bert_f1.append(f1_score)
    bert_precision.append(precision_score)
    bert_recall.append(recall_score)

    print(f"BLEU score: {bleu_score}")
    print(f"ROUGE score: {rouge_score}")
    print(f"BERT F1 score: {f1_score}")
    print(f"BERT Precision score: {precision_score}")
    print(f"BERT Recall score: {recall_score}")

    with open(f"transformers-aventIQ-{SCRIPT_LIMIT_TOKEN_SIZE}-{GENERATE_LIMIT_TOKEN_SIZE}.csv", "a") as f:
        f.write(f"{seed_text};{candidate};{reference};{bleu_score};{rouge_score};{f1_score};{precision_score};{recall_score}\n")

    i += 1

print(f"Average BLEU score : {sum(bleu_scores) / len(bleu_scores):.6f}")
print(f"Average ROUGE score : {sum(rouge_scores) / len(rouge_scores):.6f}")
print(f"Average BERT F1 score : {sum(bert_f1) / len(bert_f1):.6f}")
print(f"Average BERT Precision score : {sum(bert_precision) / len(bert_precision):.6f}")
print(f"Average BERT Recall score : {sum(bert_recall) / len(bert_recall):.6f}")


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Evaluating the model on the test set...
Evaluating 1/200...
Seed: 1 NEWLINE_TOKEN Delhi flight ! NEWLINE_TOKEN Sorry , Ma'am . NEWLINE_TOKEN Boarding is closed . NEWLINE_TOKEN Listen , I know ! NEWLINE_TOKEN I am so sorry , I got delayed . NEWLINE_TOKEN Just do something . NEWLINE_TOKEN Please help me . NEWLINE_TOKEN Sorry , Ma'am
Candidate: . NEWLINE_TOKEN Please help me . NEWLINE_TOKEN I have a daughter and she needs me . NEWLINE_TOKEN I have a beautiful son . NEWLINE_TOKEN He is very bright and cheerful . NEWLINE_
Reference: . NEWLINE_TOKEN You should have been on time . NEWLINE_TOKEN I know , I am so sorry ! NEWLINE_TOKEN I got stuck in traffic . NEWLINE_TOKEN You know how it is . NEWLINE_TOKEN Just , please ! Can you do this ? NEWLINE_TOKEN Okay , hold on . NEWLINE_TOKEN Thank
BLEU score: 0.04262595673022913
ROUGE score: 0.15999999528800013
BERT F1 score: 0.5058519840240479
BERT Precision score: 0.5406272411346436
BERT Recall score: 0.4704921245574951
Evaluating 2/200...
Seed: He 

## Hugging face Transformer (DistilGPT2)

https://huggingface.co/yroshan/moviescript

In [48]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "yroshan/moviescript"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)

In [49]:
def preprocess_text(text):
    text = re.sub(r"\n+", "\n", text)
    text = text.replace("\n", " NEWLINE_TOKEN ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def tokenize_function(example):
    processed = [preprocess_text(script) for script in example["Script"]]
    encodings = tokenizer(processed, truncation=True, padding="longest", max_length=256)
    encodings["labels"] = encodings["input_ids"].copy()
    return encodings

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["Script"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


100%|██████████| 1/1 [00:03<00:00,  3.33s/ba]


In [51]:
# Train the model (Arguments)
training_args = TrainingArguments(
    output_dir="./distil-gpt2-script",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    num_train_epochs=10,
    logging_dir="./logs",
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [52]:
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator
)


/tmp/ipykernel_1635/4126412472.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [53]:
# Train the model
trainer.train()

# Save the model
trainer.save_model("./distil-gpt2-script")

Epoch,Training Loss,Validation Loss
1,2.354100,2.180957
2,2.031800,2.187853
3,1.900700,2.218943
4,1.751600,2.269925
5,1.679200,2.316365
6,1.568500,2.381141
7,1.514900,2.422518
8,1.471500,2.475901
9,1.389800,2.511844
10,1.366800,2.528688


In [54]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import evaluate
import re
from bert_score import BERTScorer

# Load tokenizer and model
model = GPT2LMHeadModel.from_pretrained("./distil-gpt2-script")
tokenizer = GPT2Tokenizer.from_pretrained("./distil-gpt2-script")
model.eval()

# Metrics
scorer = BERTScorer(lang="en", model_type="roberta-base", rescale_with_baseline=True)

# Generation function
def generate_text(prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.9,
        top_k=50,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [56]:
import re
from bert_score import BERTScorer

bleu_scores, rouge_scores = [], []
bert_f1, bert_precision, bert_recall = [], [], []

SCRIPT_LIMIT_TOKEN_SIZE = 50
GENERATE_LIMIT_TOKEN_SIZE = 50

total = len(test_texts)
i = 1

scorer = BERTScorer(lang="en", model_type="roberta-base", rescale_with_baseline=True)

print("Evaluating the model on the test set...")
for full_text in test_texts:
    print(f"Evaluating {i}/{total}...")

    tokenized = tokenizer(full_text, return_tensors="pt", truncation=True,
                          max_length=SCRIPT_LIMIT_TOKEN_SIZE + GENERATE_LIMIT_TOKEN_SIZE)
    input_ids = tokenized.input_ids[0]

    if len(input_ids) <= SCRIPT_LIMIT_TOKEN_SIZE:
        i += 1
        continue

    seed_input_ids = input_ids[:SCRIPT_LIMIT_TOKEN_SIZE]
    expected_output_ids = input_ids[SCRIPT_LIMIT_TOKEN_SIZE:SCRIPT_LIMIT_TOKEN_SIZE + GENERATE_LIMIT_TOKEN_SIZE]

    seed_text = tokenizer.decode(seed_input_ids, skip_special_tokens=True)
    seed_text = custom_tokenize(seed_text)
    seed_text = " ".join(seed_text)
    
    reference_text = tokenizer.decode(expected_output_ids, skip_special_tokens=True).strip()
    reference_text = custom_tokenize(reference_text)
    reference_text = " ".join(reference_text)

    generated_text = generate_text(seed_text, max_new_tokens=GENERATE_LIMIT_TOKEN_SIZE)
    generated_text = re.sub(r"\s+", " ", generated_text).strip()
    generated_text = custom_tokenize(generated_text)
    generated_text = " ".join(generated_text)[len(seed_text) + 1:]

    candidate = generated_text
    reference = reference_text

    print(f"Seed: {seed_text}")
    print(f"Candidate: {candidate}")
    print(f"Reference: {reference}")
    
    bleu_score, rouge_score = calculate_scores(reference, candidate)

    P, R, F1 = scorer.score([candidate], [reference])
    f1_score = F1.mean().item()
    precision_score = P.mean().item()
    recall_score = R.mean().item()

    bleu_scores.append(bleu_score)
    rouge_scores.append(rouge_score)
    bert_f1.append(f1_score)
    bert_precision.append(precision_score)
    bert_recall.append(recall_score)

    print(f"BLEU score: {bleu_score}")
    print(f"ROUGE score: {rouge_score}")
    print(f"BERT F1 score: {f1_score}")
    print(f"BERT Precision score: {precision_score}")
    print(f"BERT Recall score: {recall_score}")

    with open(f"transformers-distilGPT2-{SCRIPT_LIMIT_TOKEN_SIZE}-{GENERATE_LIMIT_TOKEN_SIZE}.csv", "a") as f:
        f.write(f"{seed_text};{candidate};{reference};{bleu_score};{rouge_score};{f1_score};{precision_score};{recall_score}\n")

    i += 1

print(f"Average BLEU score : {sum(bleu_scores) / len(bleu_scores):.6f}")
print(f"Average ROUGE score : {sum(rouge_scores) / len(rouge_scores):.6f}")
print(f"Average BERT F1 score : {sum(bert_f1) / len(bert_f1):.6f}")
print(f"Average BERT Precision score : {sum(bert_precision) / len(bert_precision):.6f}")
print(f"Average BERT Recall score : {sum(bert_recall) / len(bert_recall):.6f}")


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Evaluating the model on the test set...
Evaluating 1/200...
Seed: 1 NEWLINE_TOKEN Delhi flight ! NEWLINE_TOKEN Sorry , Ma'am . NEWLINE_TOKEN Boarding is closed . NEWLINE_TOKEN Listen , I know ! NEWLINE_TOKEN I am so sorry , I got delayed . NEWLINE_TOKEN Just do something . NEWLINE_TOKEN Please help me . NEWLINE_TOKEN Sorry , Ma'am
Candidate: . NEWLINE_TOKEN I am so sorry . NEWLINE_TOKEN Sorry , ma'am . NEWLINE_TOKEN I know I have worked . NEWLINE_TOKEN Please help me . NEWLINE_TOKEN Please help me
Reference: . NEWLINE_TOKEN You should have been on time . NEWLINE_TOKEN I know , I am so sorry ! NEWLINE_TOKEN I got stuck in traffic . NEWLINE_TOKEN You know how it is . NEWLINE_TOKEN Just , please ! Can you do this ? NEWLINE_TOKEN Okay , hold on . NEWLINE_TOKEN Thank
BLEU score: 0.11008265656260736
ROUGE score: 0.35555555126913585
BERT F1 score: 0.617845892906189
BERT Precision score: 0.6659048795700073
BERT Recall score: 0.5698670148849487
Evaluating 2/200...
Seed: He 's going to end corru